# 02 — Uebertragbarkeit, Fehlalarme, Zielgruppe

Ersetzt die frueheren Notebooks 02, 03 und 05. Benoetigt `fall_model.keras` aus 01.

| Teil | Frage |
|---|---|
| A | Wie gut uebertraegt das SisFall-Modell auf KFall, UMAFall, FallAllD? Erklaert der Messbereich (Clipping) den Einbruch? |
| B | Wie viele Fehlalarme im Alltag am Knoechel, und hilft N-von-M-Voting? |
| C | Verhaelt sich das Modell bei aelteren Menschen (WEDA-FALL, 77–95 J.) anders? |

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q scipy scikit-learn tensorflow openpyxl

import os, sys, shutil, time, json, pickle, zipfile
from pathlib import Path
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# fall_lib laden: zuerst GitHub, sonst Kopie auf Drive. Ohne die
# aktuelle fall_lib waeren alle Ergebnisse falsch (andere Fensterlaenge).
import urllib.request
_LIB_URL = "https://raw.githubusercontent.com/Alyelb/ML-Modell-for-Fall-detection/main/fall_lib.py"
_LIB_DRIVE = "/content/drive/MyDrive/ML_Fall_Detection/fall_lib.py"
try:
    urllib.request.urlretrieve(_LIB_URL, "/content/fall_lib.py")
    print("[fall_lib] von GitHub geladen")
except Exception as _e:
    print(f"[fall_lib] GitHub nicht erreichbar ({_e}) -- nutze Drive-Kopie")
    shutil.copy(_LIB_DRIVE, "/content/fall_lib.py")
sys.path.insert(0, "/content")
import importlib, fall_lib as fl
importlib.reload(fl)
for _f in ("new_run", "build_windows", "znorm", "load_cappimu", "load_pilot",
           "augment_orientation", "rotation_between", "voting_eval", "finetune"):
    assert hasattr(fl, _f), f"fall_lib veraltet ({_f} fehlt) -- neue Version auf Drive legen."
assert fl.WIN_LEN == 100, "falsche Fensterlaenge -- fall_lib pruefen"
print("[fall_lib] Fensterlaenge", fl.WIN_LEN, "bei", fl.FS_TARGET, "Hz")

BASE  = "/content/drive/MyDrive/ML_Fall_Detection"
fl.RESULTS_ROOT = f"{BASE}/results"
MODEL = f"{BASE}/results/fall_model.keras"        # Basismodell (Teil 01)
PILOT = f"{BASE}/Pilotdaten"
CACHE = Path(f"{BASE}/cache"); CACHE.mkdir(exist_ok=True)


def hole(name, ziel):
    """Kopiert einen Datensatz von Drive auf die lokale Platte
    (Drive ist fuer tausende Kleindateien zu langsam). Gibt None
    zurueck, wenn der Ordner fehlt."""
    src = f"{BASE}/{name}"
    if Path(ziel).exists():
        print(f"[ok] {name} bereits lokal"); return ziel
    if not Path(src).exists():
        print(f"[!] {src} nicht gefunden. Vorhanden:")
        for p in sorted(Path(BASE).iterdir()):
            print("   ", p.name)
        return None
    t = time.time(); shutil.copytree(src, ziel)
    print(f"[ok] {name} kopiert in {time.time()-t:.0f}s")
    return ziel


def cached(name, fn):
    """Geladene Trials auf Drive zwischenspeichern (Laden dauert lange)."""
    p = CACHE / f"{name}.pkl"
    if p.exists():
        print(f"[cache] {name}")
        return pickle.load(open(p, "rb"))
    r = fn(); pickle.dump(r, open(p, "wb")); return r


from tensorflow import keras
def lade_modell(pfad):
    if not Path(pfad).exists():
        raise FileNotFoundError(f"{pfad} fehlt -- vorheriges Notebook ausfuehren.")
    m = keras.models.load_model(pfad, compile=False)
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return m

print("Konfiguration:", fl.default_config())

# Teil A — Kreuzvalidierung auf externen Datensaetzen

In [ ]:
from tensorflow import keras
model = keras.models.load_model(MODEL, compile=False)
model.compile(optimizer="adam", loss="binary_crossentropy",
              metrics=["accuracy"])
print(f"Basismodell geladen: {model.count_params():,} Parameter")
run = fl.new_run("02_transfer")
ergebnisse = {}

## Transferfunktion

Einheitlicher Ablauf fuer alle Datensaetze, damit die Ergebnisse
vergleichbar bleiben.

In [ ]:
def transfer(trials, name):
    X, y, g, _ = fl.build_windows(trials, mode="standard")
    if len(y) == 0 or y.sum() == 0:
        print(f"[{name}] keine Sturzfenster — uebersprungen"); return None
    X = fl.znorm(X)
    tr, te = fl.subject_split(g, y, test_size=0.25)
    print(f"\n{name}: {len(y)} Fenster, {int(y.sum())} Stuerze, "
          f"{len(np.unique(g))} Probanden")
    zs = fl.evaluate(model, X[te], y[te], "Zero-Shot")
    ft_model = fl.finetune(model, X[tr], y[tr])
    ft = fl.evaluate(ft_model, X[te], y[te], "Fine-Tuned")
    return {"zero_shot": zs, "fine_tuned": ft,
            "n_windows": int(len(y)), "n_falls": int(y.sum())}

## KFall — unterer Ruecken, 100 Hz

In [ ]:
KF = hole("KFALL/sensor_data_new", "/content/KFall")
LAB = hole("KFALL/label_data_new", "/content/KFall_labels")
kf = fl.load_kfall(KF, label_dir=LAB)
ergebnisse["kfall"] = transfer(kf, "KFall")

## UMAFall — Taille und Knoechel, 20 Hz

In [ ]:
UMA = hole("UMAFall Fall Detection/UMAFall_Dataset_corrected",
            "/content/UMAFall")
for pos in ["WAIST", "ANKLE"]:
    ergebnisse[f"umafall_{pos.lower()}"] = transfer(
        fl.load_umafall(UMA, position=pos), f"UMAFall {pos}")

## FallAllD — Taille, 238 Hz

In [ ]:
FA = hole("FallAllD", "/content/FallAllD")
ergebnisse["fallalld_waist"] = transfer(
    fl.load_fallalld(FA, device=3), "FallAllD WAIST")

## Kontrollexperiment: Messbereich

FallAllD verwendet einen LSM9DS1 mit ±8 g, SisFall und KFall ±16 g.
Hier wird geprueft, ob das Abschneiden hoher Spitzen den
Zero-Shot-Einbruch erklaert: Die SisFall-Testdaten werden synthetisch
begrenzt und erneut bewertet.

In [ ]:
sis = fl.load_sisfall(hole("SisFall_dataset", "/content/SisFall"),
                      verbose=False)
Xs, ys, gs, _ = fl.build_windows(sis, mode="standard")
_, te_s = fl.subject_split(gs, ys, test_size=0.25)
Xs_te, ys_te = Xs[te_s], ys[te_s]

peaks = fl.acc_magnitude(Xs_te).max(axis=1)
print(f"Fenster mit Spitze > 8 g: {(peaks>8).sum()}/{len(peaks)} "
      f"({100*(peaks>8).mean():.1f}%)\n")

clip = {}
for c in [16.0, 8.0, 4.0, 2.0]:
    Xc = Xs_te.copy()
    Xc[:, :, :3] = np.clip(Xc[:, :, :3], -c, c)
    clip[f"{c:g}g"] = fl.evaluate(model, fl.znorm(Xc), ys_te,
                                  f"+-{c:g} g")
ergebnisse["clipping"] = clip

## Zusammenfassung

In [ ]:
print(f"{'Datensatz':<22} {'Zero-Shot Se':>13} {'Fine-Tuned Se':>14}")
print("-"*52)
for k, v in ergebnisse.items():
    if isinstance(v, dict) and "zero_shot" in v:
        print(f"{k:<22} {v['zero_shot']['Se']:>13.3f} "
              f"{v['fine_tuned']['Se']:>14.3f}")

fl.save_results(run, ergebnisse, config=fl.default_config(),
    notes="KFall-Sturzgrenze korrigiert auf T20-T34 (Labeldateien).")

# Teil B — Fehlalarme und zeitliche Plausibilisierung

In [ ]:
from tensorflow import keras
model = keras.models.load_model(MODEL, compile=False)
model.compile(optimizer="adam", loss="binary_crossentropy",
              metrics=["accuracy"])
run = fl.new_run("03_fehlalarme")
ergebnisse = {}

POSITION = "ANKLE"     # fussnaechste verfuegbare Position
UMA = hole("UMAFall Fall Detection/UMAFall_Dataset_corrected",
           "/content/UMAFall")
trials = fl.load_umafall(UMA, position=POSITION)

## Falsch-Positiv-Test auf fussnahen Alltagsdaten

Ausgewertet werden ausschliesslich ADL-Aufzeichnungen. Da keine
Stuerze enthalten sind, ist jede positive Klassifikation per
Definition ein Fehlalarm.

In [ ]:
adl = [t for t in trials if t["label"] == 0]
X, y, g, acts = fl.build_windows(adl, mode="sliding")
X = fl.znorm(X)
probs = model.predict(X, batch_size=256, verbose=0).flatten()

stunden = len(X) * fl.WIN_SEC * (1 - fl.OVERLAP) / 3600
print(f"{len(X)} ADL-Fenster = {stunden:.2f} Stunden\n")
print(f"{'Schwelle':>9} {'FP':>7} {'Rate':>8} {'FP/Std':>9}")
print("-"*36)
schwellen = {}
for thr in [0.5, 0.7, 0.9, 0.95, 0.99]:
    fp = int((probs > thr).sum())
    print(f"{thr:>9.2f} {fp:>7d} {100*fp/len(probs):>7.2f}% "
          f"{fp/stunden:>9.1f}")
    schwellen[str(thr)] = {"fp": fp, "rate": round(fp/len(probs), 5),
                           "fp_per_hour": round(fp/stunden, 2)}
ergebnisse["fp_schwellen"] = schwellen

### Aufschluesselung nach Aktivitaet

In [ ]:
print(f"{'Aktivitaet':<26} {'Fenster':>8} {'FP':>6} {'Rate':>8} {'max p':>7}")
print("-"*60)
nach_akt = {}
for a in sorted(np.unique(acts)):
    p = probs[acts == a]
    fp = int((p > 0.5).sum())
    print(f"{a:<26} {len(p):>8d} {fp:>6d} {100*fp/len(p):>7.2f}% "
          f"{p.max():>7.3f}")
    nach_akt[a] = {"windows": int(len(p)), "fp": fp,
                   "rate": round(fp/len(p), 4),
                   "max_prob": round(float(p.max()), 4)}
ergebnisse["fp_nach_aktivitaet"] = nach_akt

## N-von-M-Voting

Ein Alarm wird erst ausgeloest, wenn innerhalb der letzten M
Entscheidungen mindestens N positiv ausfallen. Die Schrittweite
betraegt hier 0,2 s (90 % Ueberlappung), damit der Aufprall in
mehreren aufeinanderfolgenden Fenstern erscheint.

In [ ]:
STEP_FEIN = int(fl.FS_TARGET * 0.2)      # 10 Abtastwerte = 0,2 s
SEC_FEIN  = 0.2

def trial_probs(m, trials, step):
    """Wahrscheinlichkeitsfolgen je Trial, Reihenfolge erhalten."""
    seqs, fenster = [], []
    for t in trials:
        w = fl.window_sliding(t["data"], fl.WIN_LEN, step)
        if not w: continue
        fenster.append((fl.znorm(np.asarray(w, np.float32)),
                        bool(t["label"])))
    alle = np.concatenate([f for f, _ in fenster], axis=0)
    p = m.predict(alle, batch_size=512, verbose=0).flatten()
    off = 0
    for f, is_fall in fenster:
        seqs.append((p[off:off+len(f)], is_fall)); off += len(f)
    return seqs

print("Berechne Wahrscheinlichkeitsfolgen ...")
seqs_basis = trial_probs(model, trials, STEP_FEIN)
print(f"{len(seqs_basis)} Trials")

### Voting braucht ein angepasstes Modell

Verglichen wird das unveraenderte Basismodell mit einem auf
Knoecheldaten feinabgestimmten Modell. Der Unterschied zeigt, ob
Voting die Domaenenanpassung ersetzen kann.

In [ ]:
# Fine-Tuning auf einem Teil der Probanden
Xa, ya, ga, _ = fl.build_windows(trials, mode="sliding")
Xa = fl.znorm(Xa)
tr_i, te_i = fl.subject_split(ga, ya, test_size=0.3)
test_subs = set(np.unique(ga[te_i]))

model_ft = fl.finetune(model, Xa[tr_i], ya[tr_i])
print("Fine-Tuning abgeschlossen")

test_trials = [t for t in trials if t["subject"] in test_subs]
seqs = {"basis": trial_probs(model, test_trials, STEP_FEIN),
        "ft":    trial_probs(model_ft, test_trials, STEP_FEIN)}
print(f"Testsatz: {len(test_trials)} Trials")

In [ ]:
regeln = [(1,1), (2,3), (3,5), (5,7), (7,10)]
voting = {}
for tag, name in [("basis","Basismodell"), ("ft","Feinabgestimmt")]:
    print(f"\n{name}")
    print(f"  {'Regel':>9} {'Schwelle':>9} {'Se':>7} {'FP/Std':>9}")
    print("  " + "-"*38)
    for n, m_ in regeln:
        for thr in [0.5, 0.9]:
            r = fl.voting_eval(seqs[tag], thr, n, m_, SEC_FEIN)
            print(f"  {r['rule']:>9} {thr:>9.2f} "
                  f"{(r['Se'] if r['Se'] is not None else 0):>7.3f} "
                  f"{(r['fp_per_hour'] or 0):>9.1f}")
            voting[f"{tag}_{n}of{m_}@{thr}"] = r
ergebnisse["voting"] = voting

In [ ]:
fl.save_results(run, ergebnisse, config=fl.default_config(),
    notes=("Voting mit 0,2 s Schrittweite; Sturzfenster nur dort, wo "
           "der Aufprall liegt. Hard-Negative-Versuch: siehe 03_progressiv, Teil B."))

# Teil C — Zielgruppenvalidierung mit WEDA-FALL

## Datensatz kopieren

Nur der 50-Hz-Zweig wird kopiert. Nach dem Kopieren wird geprueft,
ob die erwartete Zahl an Dateien angekommen ist — ein unvollstaendiger
Kopiervorgang faellt sonst erst viel spaeter und mit unklarer
Fehlermeldung auf.

In [ ]:
QUELLE = f"{BASE}/WEDA-FALL-main/dataset/50Hz"
ZIEL   = "/content/WEDA50"

if not Path(QUELLE).exists():
    raise FileNotFoundError(
        f"{QUELLE} nicht gefunden.\n"
        f"  Inhalt von {BASE}/WEDA-FALL-main/dataset:\n    "
        + "\n    ".join(p.name for p in
                        sorted(Path(f"{BASE}/WEDA-FALL-main/dataset").iterdir())))

# Unvollstaendige Reste einer frueheren Ausfuehrung entfernen
if Path(ZIEL).exists():
    n_alt = len(list(Path(ZIEL).glob("*/*.csv")))
    print(f"[i] {ZIEL} existiert bereits mit {n_alt} CSV-Dateien")
    if n_alt < 3000:
        print("[i] unvollstaendig -> wird neu kopiert")
        shutil.rmtree(ZIEL)

if not Path(ZIEL).exists():
    print("Kopiere 50-Hz-Daten ...")
    t = time.time()
    shutil.copytree(QUELLE, ZIEL)
    print(f"  fertig in {time.time()-t:.0f}s")

# Kontrolle
ordner = sorted(p.name for p in Path(ZIEL).iterdir() if p.is_dir())
n_csv  = len(list(Path(ZIEL).glob("*/*.csv")))
print(f"\nBewegungsordner ({len(ordner)}): {ordner}")
print(f"CSV-Dateien: {n_csv}")

if len(ordner) < 19 or n_csv < 3000:
    raise RuntimeError(
        f"Kopie unvollstaendig: {len(ordner)} Ordner / {n_csv} Dateien.\n"
        f"  Erwartet: 19 Ordner (D01-D11, F01-F08), rund 3900 Dateien.\n"
        f"  Zelle erneut ausfuehren — Drive bricht bei vielen "
        f"Kleindateien gelegentlich ab.")
print("Kopie vollstaendig.")

## Modell laden

In [ ]:
from tensorflow import keras

if not Path(MODEL).exists():
    raise FileNotFoundError(
        f"{MODEL} fehlt — bitte zuerst 01_basismodelle ausfuehren.")

model = keras.models.load_model(MODEL, compile=False)
model.compile(optimizer="adam", loss="binary_crossentropy",
              metrics=["accuracy"])
print(f"Modell geladen: {model.count_params():,} Parameter")

run = fl.new_run("05_zielgruppe")
ergebnisse = {}

## Trials laden

`find_weda_trials` sucht zunaechst einen Unterordner `50Hz`. Da hier
bereits direkt auf diesen Ordner gezeigt wird, greift die
Rueckfallebene und der Ordner selbst dient als Wurzel.

In [ ]:
trials_alle = fl.find_weda_trials(ZIEL)
if not trials_alle:
    raise RuntimeError(
        f"Keine Trials erkannt. Erwartetes Schema: "
        f"<Bewegungscode>/U01_R01_accel.csv")

from collections import Counter
print(f"Trials gesamt: {len(trials_alle)}")
print(f"  nach Gruppe:  {dict(Counter(e['group'] for e in trials_alle))}")
print(f"  nach Klasse:  "
      f"{dict(Counter('Sturz' if e['label'] else 'ADL' for e in trials_alle))}")
print("\nErwartet laut Dokumentation: 350 Sturz / 619 ADL, "
      "davon 157 Trials von aelteren Probanden.")

In [ ]:
alt  = fl.load_weda(ZIEL, group="elderly")
jung = fl.load_weda(ZIEL, group="young")

if not alt or not jung:
    raise RuntimeError(
        f"Eine Gruppe ist leer (alt={len(alt)}, jung={len(jung)}). "
        f"Pruefen Sie die Zuordnung in fl.WEDA_ELDERLY / fl.WEDA_YOUNG.")

## Fairer Gruppenvergleich

Beschraenkt auf die Aktivitaeten, die beide Gruppen ausgefuehrt
haben.

In [ ]:
akt_alt   = {t["activity"] for t in alt}
akt_jung  = {t["activity"] for t in jung}
gemeinsam = sorted(akt_alt & akt_jung)

print(f"Aeltere ({len(akt_alt)}):   {sorted(akt_alt)}")
print(f"Junge   ({len(akt_jung)}):  {sorted(akt_jung)}")
print(f"Gemeinsam ({len(gemeinsam)}): {gemeinsam}")

if not gemeinsam:
    raise RuntimeError("Keine gemeinsamen Aktivitaeten — "
                       "Vergleich nicht moeglich.")

SEK_PRO_FENSTER = fl.WIN_SEC * (1 - fl.OVERLAP)

def fp_analyse(trials, aktivitaeten, name):
    """Fehlalarmrate auf ADL-Aufzeichnungen der gegebenen
    Aktivitaeten. Da keine Stuerze enthalten sind, ist jede positive
    Klassifikation ein Fehlalarm."""
    sub = [t for t in trials
           if t["activity"] in aktivitaeten and t["label"] == 0]
    if not sub:
        raise ValueError(f"{name}: keine passenden ADL-Trials")

    X, y, g, a = fl.build_windows(sub, mode="sliding")
    if len(X) == 0:
        raise ValueError(f"{name}: keine Fenster erzeugt")

    p = model.predict(fl.znorm(X), batch_size=256, verbose=0).flatten()
    h = len(X) * SEK_PRO_FENSTER / 3600
    print(f"  {name:<10} {len(X):>6} Fenster = {h:>5.2f} Std, "
          f"{len(np.unique(g)):>2} Probanden")
    return p, a, h

print()
p_alt,  a_alt,  h_alt  = fp_analyse(alt,  gemeinsam, "Aeltere")
p_jung, a_jung, h_jung = fp_analyse(jung, gemeinsam, "Junge")

### Fehlalarmrate nach Entscheidungsschwelle

In [ ]:
print(f"{'Schwelle':>9} {'FP alt':>8} {'/Std':>8} "
      f"{'FP jung':>9} {'/Std':>8} {'Verhaeltnis':>12}")
print("-"*60)

vergleich = {}
for thr in [0.5, 0.7, 0.9, 0.95, 0.99]:
    fa, fj = int((p_alt > thr).sum()), int((p_jung > thr).sum())
    ra, rj = fa / h_alt, fj / h_jung
    q = ra / rj if rj > 0 else float("nan")
    print(f"{thr:>9.2f} {fa:>8d} {ra:>8.1f} {fj:>9d} {rj:>8.1f} "
          f"{q:>11.2f}x")
    vergleich[str(thr)] = {
        "elderly_fp": fa, "elderly_per_hour": round(ra, 2),
        "young_fp": fj,   "young_per_hour": round(rj, 2),
        "ratio": round(float(q), 3)}

ergebnisse["fairer_vergleich"] = {
    "aktivitaeten": gemeinsam,
    "elderly_hours": round(h_alt, 3),
    "young_hours": round(h_jung, 3),
    "schwellen": vergleich}

print("\nEin deutlich erhoehtes Verhaeltnis wuerde bedeuten, dass "
      "Alltagsbewegungen\naelterer Menschen fuer das auf jungen "
      "Kohorten trainierte Modell schwerer\nzu unterscheiden sind.")

### Aufschluesselung nach Bewegung

Zeigt, ob ein etwaiger Gruppenunterschied auf einzelne Bewegungen
zurueckgeht oder durchgaengig auftritt.

In [ ]:
print(f"{'Code':<6} {'Bewegung':<24} {'alt %':>8} {'jung %':>8} "
      f"{'Differenz':>11}")
print("-"*62)

nach_bewegung = {}
for mv in gemeinsam:
    pa, pj = p_alt[a_alt == mv], p_jung[a_jung == mv]
    if len(pa) == 0 or len(pj) == 0:
        continue
    ra, rj = 100 * (pa > 0.9).mean(), 100 * (pj > 0.9).mean()
    print(f"{mv:<6} {fl.WEDA_MOVEMENTS.get(mv, mv):<24} "
          f"{ra:>7.2f}% {rj:>7.2f}% {ra-rj:>+10.2f}pp")
    nach_bewegung[mv] = {
        "name": fl.WEDA_MOVEMENTS.get(mv, mv),
        "elderly_rate": round(ra / 100, 4),
        "young_rate": round(rj / 100, 4),
        "n_elderly": int(len(pa)), "n_young": int(len(pj))}

ergebnisse["nach_bewegung"] = nach_bewegung

## Sensitivitaet bei den jungen Probanden

Nur diese Gruppe hat Stuerze ausgefuehrt. Der Wert dient als
Einordnung der Fehlalarmzahlen — eine niedrige Fehlalarmrate bei
gleichzeitig niedriger Sensitivitaet waere kein gutes Ergebnis.

In [ ]:
X_j, y_j, g_j, a_j = fl.build_windows(jung, mode="standard")
if y_j.sum() > 0:
    ergebnisse["junge_gesamt"] = fl.evaluate(
        model, fl.znorm(X_j), y_j, "Junge (alle Aktivitaeten)")
    n_f = int(y_j.sum())
    print(f"\n  {n_f} Sturzfenster von "
          f"{len({t['subject'] for t in jung if t['label']==1})} Probanden")
else:
    print("Keine Sturzfenster vorhanden.")

## Ergebnisse sichern

In [ ]:
fl.save_results(
    run, ergebnisse, config=fl.default_config(),
    notes=("Aeltere Probanden ohne Stuerze -> nur Fehlalarmrate "
           "bewertbar. Gruppenvergleich auf gemeinsame Aktivitaeten "
           "beschraenkt. Nur 50-Hz-Zweig verwendet; Fitbit-"
           "Zeitstempel auf regelmaessiges Raster interpoliert."))

print()
print(json.dumps(ergebnisse.get("fairer_vergleich", {}), indent=2)[:1200])